# Module 3.0: AI Search Index Setup

This notebook creates the Azure AI Search index that serves as the **RAG knowledge base**
for the travel agent. It loads all corporate policy documents into a searchable index
with vector embeddings for hybrid (keyword + semantic) retrieval.

## Why AI Search?

RAG is the **authoritative ground truth** in our architecture:
- Policies are maintained by policy teams → indexed in AI Search
- Memory is the personalised layer that must stay **consistent** with RAG
- When memory contradicts RAG, memory is wrong (→ Notebook 07: Procedural Lifecycle)

## Prerequisites

1. Azure AI Search resource provisioned (any tier — Free works for this demo)
2. Azure OpenAI with an embedding model deployed (text-embedding-3-small recommended)
3. Environment variables set:
   - `AZURE_SEARCH_ENDPOINT` — e.g. `https://your-search.search.windows.net`
   - `AZURE_SEARCH_KEY` — Admin key (or use managed identity)
   - `FOUNDRY_PROJECT_ENDPOINT` — for embeddings
   - `EMBEDDING_MODEL` — deployment name (default: `text-embedding-3-small`)

In [ ]:
%pip install -q azure-search-documents azure-identity openai

In [ ]:
import sys, os, json
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from azure.identity import AzureCliCredential

credential = AzureCliCredential()

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Embedding model: {EMBEDDING_MODEL}")

## Step 1: Define the Index Schema

Each document in the index has:
- `id` — unique document identifier
- `title` — document title for display
- `content` — full text (searchable)
- `category` — document type for filtering
- `version` — version string for change detection
- `last_updated` — when the document was last modified
- `content_vector` — embedding for semantic search

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
    SearchField,
)

INDEX_NAME = "travel-policies"
VECTOR_DIMENSIONS = 1536  # text-embedding-3-small

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=credential,
)

# Define fields
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="title", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="version", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="last_updated", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=VECTOR_DIMENSIONS,
        vector_search_profile_name="default-profile",
    ),
]

# Vector search configuration
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="default-algorithm")],
    profiles=[VectorSearchProfile(name="default-profile", algorithm_configuration_name="default-algorithm")],
)

# Semantic configuration
semantic_config = SemanticConfiguration(
    name="default-semantic",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        content_fields=[SemanticField(field_name="content")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create or update the index
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created/updated successfully")
print(f"  Fields: {len(result.fields)}")
print(f"  Vector search: {result.vector_search is not None}")
print(f"  Semantic search: {result.semantic_search is not None}")

## Step 2: Prepare Documents for Indexing

We load all policy documents from `data/policies/` and the existing skill files,
then structure them for upload.

In [ ]:
POLICIES_DIR = Path("../data/policies")
SKILLS_DIR = Path("../02_memory_layers/skills")


def load_document(filepath: Path, doc_id: str, title: str, category: str,
                  version: str = "1.0") -> dict:
    """Load a file and structure it as a search document."""
    content = filepath.read_text(encoding="utf-8")
    # For JSON files, convert to readable text
    if filepath.suffix == ".json":
        data = json.loads(content)
        content = json.dumps(data, indent=2)
    return {
        "id": doc_id,
        "title": title,
        "content": content,
        "category": category,
        "version": version,
        "last_updated": datetime.now(timezone.utc).isoformat(),
    }


# Collect all documents
documents = [
    # Policy documents
    load_document(
        POLICIES_DIR / "general_travel_policy.md",
        "general-travel-policy",
        "General Travel Policy",
        "policy",
        version="4.2",
    ),
    load_document(
        POLICIES_DIR / "expense_reimbursement.md",
        "expense-reimbursement",
        "Expense Reimbursement Policy",
        "policy",
        version="2.1",
    ),
    load_document(
        POLICIES_DIR / "preferred_vendors.md",
        "preferred-vendors",
        "Preferred Vendors Policy",
        "policy",
        version="3.0",
    ),
    load_document(
        POLICIES_DIR / "travel_safety.md",
        "travel-safety",
        "Travel Safety & Compliance Policy",
        "compliance",
        version="1.4",
    ),
    load_document(
        POLICIES_DIR / "per_diem_rates.json",
        "per-diem-rates",
        "Per-Diem Rates by City",
        "policy",
        version="2026-Q3",
    ),
    # Existing skill/procedure documents
    load_document(
        SKILLS_DIR / "domestic-booking" / "SKILL.md",
        "domestic-booking-procedure",
        "Domestic Booking Procedure",
        "procedure",
    ),
    load_document(
        SKILLS_DIR / "international-booking" / "SKILL.md",
        "international-booking-procedure",
        "International Booking Procedure",
        "procedure",
    ),
    load_document(
        SKILLS_DIR / "international-booking" / "visa-checklist.md",
        "visa-checklist",
        "Visa Requirements Checklist",
        "compliance",
    ),
    load_document(
        SKILLS_DIR / "domestic-booking" / "budget-limits.json",
        "budget-limits-domestic",
        "Budget Limits — Domestic Travel",
        "policy",
    ),
    load_document(
        SKILLS_DIR / "international-booking" / "budget-limits.json",
        "budget-limits-international",
        "Budget Limits — International Travel",
        "policy",
    ),
]

print(f"Prepared {len(documents)} documents for indexing:")
print()
for doc in documents:
    print(f"  [{doc['category']:<10}] {doc['id']:<35} v{doc['version']}")

## Step 3: Generate Embeddings

We generate vector embeddings for each document's content using the Azure OpenAI
embedding model. These enable semantic (meaning-based) search.

In [ ]:
from openai import AzureOpenAI

# Use Azure OpenAI for embeddings
openai_client = AzureOpenAI(
    azure_endpoint=FOUNDRY_ENDPOINT,
    azure_ad_token_provider=lambda: credential.get_token(
        "https://cognitiveservices.azure.com/.default"
    ).token,
    api_version="2024-10-21",
)


def get_embedding(text: str) -> list[float]:
    """Generate embedding for a text string."""
    # Truncate to model limit if needed (8191 tokens for text-embedding-3-small)
    response = openai_client.embeddings.create(
        input=text[:8000],  # Safe character limit
        model=EMBEDDING_MODEL,
    )
    return response.data[0].embedding


# Generate embeddings for all documents
print("Generating embeddings...")
for i, doc in enumerate(documents):
    doc["content_vector"] = get_embedding(doc["content"])
    print(f"  [{i+1}/{len(documents)}] {doc['id']} — {len(doc['content_vector'])} dimensions")

print(f"\nAll {len(documents)} embeddings generated")

## Step 4: Upload Documents to Index

Upload all documents with their embeddings to the AI Search index.

In [ ]:
from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential,
)

# Upload documents
result = search_client.upload_documents(documents=documents)

succeeded = sum(1 for r in result if r.succeeded)
failed = sum(1 for r in result if not r.succeeded)
print(f"Upload complete: {succeeded} succeeded, {failed} failed")

if failed:
    for r in result:
        if not r.succeeded:
            print(f"  FAILED: {r.key} — {r.error_message}")

## Step 5: Test the Index

Let's verify the index works with both keyword and hybrid (keyword + vector) search.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def search_policies(query: str, top_k: int = 3, category: str = None) -> list[dict]:
    """Search the policy index with hybrid (keyword + vector) search."""
    vector_query = VectorizedQuery(
        vector=get_embedding(query),
        k_nearest_neighbors=top_k,
        fields="content_vector",
    )

    filter_expr = f"category eq '{category}'" if category else None

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        filter=filter_expr,
        top=top_k,
        select=["id", "title", "content", "category", "version", "last_updated"],
    )

    return [
        {
            "id": r["id"],
            "title": r["title"],
            "content": r["content"][:500],  # Truncate for display
            "category": r["category"],
            "version": r["version"],
            "score": r["@search.score"],
        }
        for r in results
    ]


print("search_policies() function ready")

In [ ]:
# Test queries
test_queries = [
    "What is the hotel budget limit for a Senior employee?",
    "Which hotel chain should I book?",
    "What is the per-diem rate for New York?",
    "Do I need a visa for international travel?",
    "What expenses can I claim for reimbursement?",
]

for query in test_queries:
    results = search_policies(query, top_k=2)
    print(f"\nQ: {query}")
    for r in results:
        print(f"  → [{r['category']}] {r['title']} (v{r['version']}, score={r['score']:.3f})")

## Step 6: Reusable Search Tool

This tool function can be used by other notebooks and agents. It wraps the
hybrid search into an `@tool`-compatible function.

In [ ]:
from agent_framework import tool


@tool
async def search_travel_policies(
    query: str, category: str = "", top_k: int = 3
) -> str:
    """Search corporate travel policies, procedures, and compliance documents.
    
    Use this to find current policy information about budgets, vendors,
    safety requirements, expense rules, and booking procedures.
    
    Args:
        query: Natural language question about travel policies
        category: Optional filter — 'policy', 'procedure', or 'compliance'
        top_k: Number of results to return (default 3)
    """
    results = search_policies(query, top_k=top_k, category=category or None)
    if not results:
        return "No matching policies found."
    return json.dumps(results, indent=2)


print("@tool search_travel_policies ready")
print("This tool can be added to any agent's tool list for RAG-based policy lookup.")

## Step 7: Verify Full-Text Retrieval

For procedural lifecycle validation (Notebook 07), we need to retrieve the
**full document content** for comparison against stored memories. Let's verify
we can fetch complete policy documents by ID.

In [ ]:
def get_policy_by_id(doc_id: str) -> dict | None:
    """Retrieve a specific policy document by ID (for version comparison)."""
    try:
        result = search_client.get_document(key=doc_id)
        return {
            "id": result["id"],
            "title": result["title"],
            "content": result["content"],
            "version": result["version"],
            "last_updated": result["last_updated"],
        }
    except Exception:
        return None


# Test: get the preferred vendors policy
vendor_policy = get_policy_by_id("preferred-vendors")
if vendor_policy:
    print(f"Retrieved: {vendor_policy['title']}")
    print(f"Version:   {vendor_policy['version']}")
    print(f"Updated:   {vendor_policy['last_updated']}")
    print(f"Content:   {len(vendor_policy['content'])} characters")
    print(f"\nFirst 200 chars:")
    print(f"  {vendor_policy['content'][:200]}...")

## Summary

The AI Search index is now set up with:

| Document | Category | Version |
|----------|----------|----------|
| General Travel Policy | policy | 4.2 |
| Expense Reimbursement | policy | 2.1 |
| Preferred Vendors | policy | 3.0 |
| Travel Safety & Compliance | compliance | 1.4 |
| Per-Diem Rates | policy | 2026-Q3 |
| Domestic Booking Procedure | procedure | 1.0 |
| International Booking Procedure | procedure | 1.0 |
| Visa Checklist | compliance | 1.0 |
| Budget Limits (Domestic) | policy | 1.0 |
| Budget Limits (International) | policy | 1.0 |

### Available functions:
- `search_policies(query, top_k, category)` — hybrid search for notebooks
- `search_travel_policies(query, category, top_k)` — `@tool` for agents
- `get_policy_by_id(doc_id)` — direct retrieval for version comparison

### Next notebooks:
- **01_memory_vs_rag.ipynb** — uses `search_travel_policies` as the RAG tool
- **07_procedural_lifecycle.ipynb** — uses `get_policy_by_id` + `search_policies` to validate stored reflections against current policy